# ETL Notebook 3: Trips Data Transformation
Local pandas version for Windows development.
Transforms bronze data for ML: joins vehicle info, preprocesses, predicts fuel, computes KPIs.

## Imports

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import os
import warnings
warnings.filterwarnings('ignore')

# For ML model placeholder
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

## Configuration

In [ ]:
# Configuration
ARTIFACT_PATH = "./artifacts"

# File paths
BRONZE_TABLE_FILE = os.path.join(ARTIFACT_PATH, "trips_bronze.parquet")
VEHICLE_INFO_FILE = os.path.join(ARTIFACT_PATH, "vehicle_info.parquet")
SILVER_TABLE_FILE = os.path.join(ARTIFACT_PATH, "trips_silver.parquet")
METRICS_FILE = os.path.join(ARTIFACT_PATH, "metrics.parquet")

# Parameters
RUN_DATE = "2025-12-15"
ML_MODEL_PLACEHOLDER = True  # Use placeholder model since we don't have MLflow locally

# Create artifact directory
os.makedirs(ARTIFACT_PATH, exist_ok=True)

print(f"Configuration:")
print(f"  Bronze input: {BRONZE_TABLE_FILE}")
print(f"  Vehicle info: {VEHICLE_INFO_FILE}")
print(f"  Silver output: {SILVER_TABLE_FILE}")
print(f"  Run date: {RUN_DATE}")

## Step 1: Start Timer

In [ ]:
transformation_start_ts = datetime.utcnow()
print(f"Transformation started at: {transformation_start_ts}")

## Step 2: Load Bronze Data

In [ ]:
# Read bronze trips data
bronze_df = pd.read_parquet(BRONZE_TABLE_FILE)
print(f"Loaded {len(bronze_df)} records from bronze layer")
print(f"Columns: {list(bronze_df.columns)}")

## Step 3: Load Vehicle Info

In [ ]:
# Load vehicle metadata (must exist for this notebook)
if os.path.exists(VEHICLE_INFO_FILE):
    vehicle_info_df = pd.read_parquet(VEHICLE_INFO_FILE)
    print(f"Loaded {len(vehicle_info_df)} vehicle records")
else:
    # Create placeholder vehicle info if file doesn't exist
    vehicle_ids = bronze_df['unit_id'].unique()
    vehicle_info_df = pd.DataFrame({
        'unit_id': vehicle_ids,
        'vehicle_class': 'sedan',
        'engine_cc': 2000,
        'fuel_type': 'petrol',
        'transmission': 'automatic',
        'model_year': 2020
    })
    print(f"Created placeholder vehicle info for {len(vehicle_info_df)} vehicles")

## Step 4: Create ML Model Placeholder

In [ ]:
def get_fuel_consumption_model():
    """
    Returns a placeholder ML model for fuel consumption prediction.
    In production, this would load from MLflow: mlflow.spark.load_model(model_uri)
    
    For local development, we use a simple scikit-learn model.
    """
    # Create a simple mock model
    # In real scenario, this would be trained on historical data
    
    class FuelConsumptionModel:
        def __init__(self):
            self.scaler = StandardScaler()
            # Fit scaler on reasonable ranges
            self.scaler.fit(np.array([[0, 0, 0, 2000],
                                       [300, 3600, 200, 5000]]))
            self.rf = RandomForestRegressor(n_estimators=10, random_state=42)
            # Dummy training data
            X_dummy = self.scaler.transform(np.array([
                [100, 1800, 50, 2000],
                [200, 3600, 100, 2500],
                [150, 2700, 75, 2200]
            ]))
            y_dummy = np.array([6.5, 8.2, 7.1])  # Fuel consumption in L/100km
            self.rf.fit(X_dummy, y_dummy)

        def predict(self, distance_km, duration_sec, avg_speed, engine_cc):
            """
            Predict fuel consumption given trip and vehicle features.
            Returns fuel_consumed in liters.
            """
            try:
                # Prepare features
                X = np.array([[distance_km, duration_sec, avg_speed, engine_cc]])
                X_scaled = self.scaler.transform(X)
                
                # Predict fuel consumption (L/100km)
                fuel_per_100km = self.rf.predict(X_scaled)[0]
                
                # Convert to actual consumed (L)
                fuel_consumed = (fuel_per_100km * distance_km) / 100.0
                return max(0, fuel_consumed)  # Ensure non-negative
            except:
                # Fallback: simple linear estimate
                return distance_km * 0.07  # ~7L per 100km baseline

    return FuelConsumptionModel()

# Initialize model
fuel_model = get_fuel_consumption_model()
print("✓ Fuel consumption model initialized")

## Step 5: Define Transformation Functions

In [ ]:
def add_vehicle_info(df, vehicle_info_df):
    """
    Join vehicle information to trips.
    """
    df = df.merge(
        vehicle_info_df,
        on='unit_id',
        how='left'
    )
    return df


def preprocess_for_ml_model(df):
    """
    Preprocess features for ML model.
    """
    df = df.copy()
    
    # Ensure required columns exist
    if 'distance_km' not in df.columns and 'distance' in df.columns:
        df['distance_km'] = df['distance'] / 1000.0
    
    if 'duration' not in df.columns and 'start' in df.columns and 'end' in df.columns:
        df['duration'] = (df['end'] - df['start']).dt.total_seconds().astype(int)
    
    if 'avg_speed' not in df.columns and 'distance_km' in df.columns and 'duration' in df.columns:
        df['avg_speed'] = np.where(
            df['duration'] > 0,
            df['distance_km'] / (df['duration'] / 3600.0),
            0
        )
    
    # Fill missing engine_cc with median
    if 'engine_cc' in df.columns:
        df['engine_cc'].fillna(df['engine_cc'].median(), inplace=True)
    
    return df


def predict_fuel_consumption(df, fuel_model):
    """
    Predict fuel consumption for each trip.
    """
    df = df.copy()
    
    # Predict fuel_consumed
    df['fuel_consumed_predicted'] = df.apply(
        lambda row: fuel_model.predict(
            row.get('distance_km', 0),
            row.get('duration', 0),
            row.get('avg_speed', 0),
            row.get('engine_cc', 2000)
        ),
        axis=1
    )
    
    # Residual analysis
    if 'fuel_consumption' in df.columns:
        df['fuel_residual'] = (
            df['fuel_consumption'] - df['fuel_consumed_predicted']
        ).abs()
    
    return df


def compute_idling_kpi(df):
    """
    Compute idling-related KPIs.
    """
    df = df.copy()
    
    if 'idle_time' in df.columns and 'duration' in df.columns:
        df['idle_percentage'] = (
            df['idle_time'] / df['duration'] * 100.0
        )
    
    if 'fuel_consumed_predicted' in df.columns and 'distance_km' in df.columns:
        df['fuel_per_100km_pred'] = (
            df['fuel_consumed_predicted'] / df['distance_km'] * 100.0
        ).replace([np.inf, -np.inf], np.nan)
    
    return df


print("✓ Transformation functions defined")

## Step 6: Add Vehicle Info

In [ ]:
silver_df = add_vehicle_info(bronze_df, vehicle_info_df)
print(f"Added vehicle info: {len(silver_df)} records")

## Step 7: Preprocess Features

In [ ]:
silver_df = preprocess_for_ml_model(silver_df)
print(f"Preprocessed for ML: {len(silver_df)} records")

## Step 8: Predict Fuel Consumption

In [ ]:
silver_df = predict_fuel_consumption(silver_df, fuel_model)
print(f"Added predictions: {len(silver_df)} records")

# Track residuals for metrics
if 'fuel_residual' in silver_df.columns:
    residual_mean = silver_df['fuel_residual'].mean()
    residual_std = silver_df['fuel_residual'].std()
    residual_p95 = silver_df['fuel_residual'].quantile(0.95)
    print(f"Residual stats: mean={residual_mean:.3f}, std={residual_std:.3f}, p95={residual_p95:.3f}")

## Step 9: Compute KPIs

In [ ]:
silver_df = compute_idling_kpi(silver_df)
print(f"Computed KPIs: {len(silver_df)} records")

## Step 10: Unit-Level Aggregations

In [ ]:
# Compute unit-level aggregated metrics
unit_agg_metrics = {}

if 'unit_id' in silver_df.columns:
    unit_groups = silver_df.groupby('unit_id')
    
    # Fuel efficiency
    if 'fuel_per_100km_pred' in silver_df.columns:
        unit_agg_metrics['silver_fuel_per_100km_mean'] = silver_df['fuel_per_100km_pred'].mean()
        unit_agg_metrics['silver_fuel_per_100km_p95'] = silver_df['fuel_per_100km_pred'].quantile(0.95)
    
    # Idling
    if 'idle_percentage' in silver_df.columns:
        unit_agg_metrics['silver_idle_percentage_mean'] = silver_df['idle_percentage'].mean()
        unit_agg_metrics['silver_idle_percentage_p95'] = silver_df['idle_percentage'].quantile(0.95)
    
    # Trip metrics
    unit_agg_metrics['silver_trips_count'] = len(silver_df)
    if 'distance_km' in silver_df.columns:
        unit_agg_metrics['silver_total_distance_km'] = silver_df['distance_km'].sum()
        unit_agg_metrics['silver_avg_trip_distance'] = silver_df['distance_km'].mean()
    
    if 'duration' in silver_df.columns:
        unit_agg_metrics['silver_total_duration_sec'] = silver_df['duration'].sum()
        unit_agg_metrics['silver_avg_trip_duration'] = silver_df['duration'].mean()

print(f"Computed {len(unit_agg_metrics)} unit-level metrics")

## Step 11: Save Silver Table

In [ ]:
silver_df.to_parquet(SILVER_TABLE_FILE, index=False)
print(f"✓ Silver table saved to: {SILVER_TABLE_FILE}")
print(f"  Shape: {silver_df.shape}")

## Step 12: Record Transformation Time & Metrics

In [ ]:
transformation_end_ts = datetime.utcnow()
transformation_duration_sec = (transformation_end_ts - transformation_start_ts).total_seconds()

unit_agg_metrics["silver_transformation_duration_sec"] = transformation_duration_sec
unit_agg_metrics["silver_output_rows"] = len(silver_df)

print(f"Transformation duration: {transformation_duration_sec:.2f} seconds")

## Step 13: Create Metrics Dataframe

In [ ]:
# Convert metrics dict to dataframe
metrics_rows = []
run_date_obj = pd.to_datetime(RUN_DATE).date()
created_at_ts = datetime.utcnow()

for metric_name, metric_value in unit_agg_metrics.items():
    metrics_rows.append({
        'date': run_date_obj,
        'pipeline_stage': 'silver',
        'metric_name': metric_name,
        'metric_value': float(metric_value) if metric_value is not None else None,
        'created_at': created_at_ts
    })

metrics_df = pd.DataFrame(metrics_rows)
print(f"Created {len(metrics_df)} metric records")

## Step 14: Save Metrics

In [ ]:
# Append to existing metrics or create new
if os.path.exists(METRICS_FILE):
    existing_metrics = pd.read_parquet(METRICS_FILE)
    metrics_df = pd.concat([existing_metrics, metrics_df], ignore_index=True)

metrics_df.to_parquet(METRICS_FILE, index=False)
print(f"✓ Metrics saved to: {METRICS_FILE}")
print(f"  Total metrics in file: {len(metrics_df)}")

## Step 15: Summary

In [ ]:
print("\n" + "="*60)
print("SILVER TRANSFORMATION COMPLETED")
print("="*60)
print(f"Date: {RUN_DATE}")
print(f"Input records (bronze): {len(bronze_df)}")
print(f"Output records (silver): {len(silver_df)}")
print(f"Duration: {transformation_duration_sec:.2f} seconds")
print(f"Output file: {SILVER_TABLE_FILE}")
print("="*60)